# 📊 XYZ Loan App — Comprehensive Data Analysis

- **Data-1:** ~238K customer complaints/issues
- **Data-2:** ~741 Google Play reviews with ratings, sentiment, and themes

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from textblob import TextBlob
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

COLORS = px.colors.qualitative.Set2
NEG_POS = {"Negative": "#EF553B", "Positive": "#00CC96", "Neutral": "#FFA15A"}
print("✅ Imports ready")

## Part 1: Data-1 — Customer Complaints

In [ ]:
df1 = pd.read_csv("csv/data-1-new.csv", encoding="utf-8-sig")
df1.columns = df1.columns.str.strip()
df1["DateTime"] = pd.to_datetime(df1["DateTime"], format="%d-%m-%Y %H.%M", errors="coerce")
df1 = df1.rename(columns={"Gender": "gender", "Age_range": "age_range", "Zone": "zone",
                           "Education_Cleaned": "education", "Monthly_Income_Cleaned": "income_bucket"})
for col in ["Query Type","Source","Category","Sub Category","gender","education","age_range","zone","income_bucket"]:
    df1[col] = df1[col].astype(str).str.strip().replace({"nan":"Unknown","":"Unknown"})
df1["month"] = df1["DateTime"].dt.to_period("M").astype(str)
print(f"✅ {len(df1):,} rows loaded")
df1.head()

In [ ]:
# Category Distribution
cat_dist = df1["Category"].value_counts().reset_index()
cat_dist.columns = ["Category","Count"]
cat_dist["Pct"] = (cat_dist["Count"]/cat_dist["Count"].sum()*100).round(2)
fig = px.pie(cat_dist, values="Count", names="Category", hole=0.4, color_discrete_sequence=COLORS,
             title="Category Distribution")
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.update_layout(height=500, showlegend=False)
fig.show()
display(cat_dist)

In [ ]:
# Top 10 Sub-Categories
sub_dist = df1["Sub Category"].value_counts().head(10).reset_index()
sub_dist.columns = ["Sub Category","Count"]
sub_dist["Pct"] = (sub_dist["Count"]/len(df1)*100).round(2)
fig = px.bar(sub_dist, x="Count", y="Sub Category", orientation="h", text="Pct",
             color="Count", color_continuous_scale="Blues", title="Top 10 Sub-Categories")
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(height=450, yaxis=dict(categoryorder="total ascending"), coloraxis_showscale=False)
fig.show()

In [ ]:
# Source Channel
src = df1["Source"].value_counts().reset_index()
src.columns = ["Source","Count"]
fig = px.pie(src, values="Count", names="Source", hole=0.45, color_discrete_sequence=COLORS,
             title="Complaint Source Channels")
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Cohort Distributions
fig = make_subplots(rows=2, cols=2, subplot_titles=("Gender","Age Range","Education","Zone"),
                    specs=[[{"type":"pie"},{"type":"pie"}],[{"type":"pie"},{"type":"pie"}]])
for i, col in enumerate(["gender","age_range","education","zone"]):
    c = df1[col].value_counts()
    r, cc = divmod(i, 2)
    fig.add_trace(go.Pie(labels=c.index, values=c.values, hole=0.35, marker_colors=COLORS,
                         textposition="outside", textinfo="percent+label"), row=r+1, col=cc+1)
fig.update_layout(height=800, showlegend=False, title_text="Cohort Distribution")
fig.show()

In [ ]:
# Category × Cohort stacked bars
for cohort in ["gender", "age_range", "education", "zone", "income_bucket"]:
    ct = pd.crosstab(df1[cohort], df1["Category"])
    ct_pct = ct.div(ct.sum(axis=1), axis=0)*100
    fig = px.bar(ct_pct.reset_index(), x=cohort, y=ct_pct.columns.tolist(),
                 barmode="stack", color_discrete_sequence=COLORS,
                 title=f"Category by {cohort.replace('_',' ').title()} (%)")
    fig.update_layout(height=400, yaxis_title="%")
    fig.show()

In [ ]:
# Top 10 Sub-Category × Cohort heatmaps
top_subs = df1["Sub Category"].value_counts().head(10).index
df_top = df1[df1["Sub Category"].isin(top_subs)]
for cohort in ["gender", "age_range", "zone"]:
    ct = pd.crosstab(df_top[cohort], df_top["Sub Category"])
    fig = px.imshow(ct.T, text_auto=True, aspect="auto", color_continuous_scale="YlOrRd",
                    title=f"Top 10 Sub-Category × {cohort.replace('_',' ').title()}")
    fig.update_layout(height=450)
    fig.show()

In [ ]:
# Monthly Trends
monthly = df1.groupby("month").size().reset_index(name="Count")
fig = px.line(monthly, x="month", y="Count", markers=True, title="Monthly Issue Volume", line_shape="spline")
fig.show()

cat_m = df1.groupby(["month","Category"]).size().reset_index(name="Count")
fig = px.line(cat_m, x="month", y="Count", color="Category", markers=True,
              color_discrete_sequence=COLORS, title="Monthly Trend by Category")
fig.update_layout(height=450)
fig.show()

---
## Part 2: Data-2 — Google Reviews

In [ ]:
df2 = pd.read_csv("csv/data-2-new.csv", encoding="utf-8-sig")
df2.columns = df2.columns.str.strip()
df2 = df2.rename(columns={"Google_Review":"content","Rating (1-5 star)":"rating","DateTime":"review_date",
                           "Sentiment":"sentiment","Age_Range":"age_range","Zone":"zone",
                           "Education":"education","Monthly_Income_Cleaned":"income_bucket","Theme":"theme"})
df2["rating"] = pd.to_numeric(df2["rating"], errors="coerce")
for col in ["sentiment","age_range","zone","education","income_bucket","theme"]:
    df2[col] = df2[col].astype(str).str.strip()

# TextBlob scores
df2["sentiment_score"] = df2["content"].apply(lambda t: TextBlob(str(t)).sentiment.polarity if isinstance(t,str) else 0)

print(f"✅ {len(df2):,} reviews loaded")
print(f"Themes: {df2['theme'].unique().tolist()}")
print(f"Sentiment: {df2['sentiment'].value_counts().to_dict()}")
df2.head()

In [ ]:
# Sentiment Scorecard
pos = (df2['sentiment']=='Positive').sum()
neg = (df2['sentiment']=='Negative').sum()
print("="*60)
print("📊 SENTIMENT SCORECARD")
print("="*60)
print(f"  Total Reviews:   {len(df2):,}")
print(f"  Avg Rating:      {df2['rating'].mean():.2f} ⭐")
print(f"  Positive:        {pos:,} ({pos/len(df2)*100:.1f}%)")
print(f"  Negative:        {neg:,} ({neg/len(df2)*100:.1f}%)")
print(f"  Avg Sentiment:   {df2['sentiment_score'].mean():.3f}")
print(f"  5-Star:          {(df2['rating']==5).sum():,} ({(df2['rating']==5).mean()*100:.1f}%)")
print("="*60)

In [ ]:
# Rating & Sentiment Split
rd = df2["rating"].value_counts().sort_index().reset_index()
rd.columns = ["Rating","Count"]
rd["Pct"] = (rd["Count"]/rd["Count"].sum()*100).round(2)
sd = df2["sentiment"].value_counts().reset_index()
sd.columns = ["Sentiment","Count"]

fig = make_subplots(rows=1, cols=2, specs=[[{"type":"bar"},{"type":"pie"}]],
                    subplot_titles=("Rating Distribution","Sentiment Split"))
fig.add_trace(go.Bar(x=rd["Rating"], y=rd["Count"],
                     marker_color=["#EF553B","#EF553B","#FFA15A","#00CC96","#00CC96"],
                     text=rd["Pct"].apply(lambda x: f"{x}%"), textposition="outside"), row=1, col=1)
fig.add_trace(go.Pie(labels=sd["Sentiment"], values=sd["Count"], hole=0.45,
                     marker_colors=[NEG_POS.get(s,"gray") for s in sd["Sentiment"]],
                     textposition="outside", textinfo="percent+label"), row=1, col=2)
fig.update_layout(height=400, showlegend=False, title_text="Rating & Sentiment")
fig.show()

In [ ]:
# Diverging Stacked Bar - Sentiment Score
bins = [-1, -0.5, -0.2, -0.05, 0.05, 0.2, 0.5, 1]
lb = ["Very Neg","Negative","Slight Neg","Neutral","Slight Pos","Positive","Very Pos"]
df2["score_bin"] = pd.cut(df2["sentiment_score"], bins=bins, labels=lb)
bc = df2["score_bin"].value_counts().reindex(lb).fillna(0)
colors_d = ["#d73027","#f46d43","#fdae61","#ffffbf","#a6d96a","#66bd63","#1a9850"]

fig = go.Figure()
for i, (label, count) in enumerate(bc.items()):
    x_val = -count if i < 3 else count
    fig.add_trace(go.Bar(y=["Sentiment"], x=[x_val], name=label, orientation="h",
                         marker_color=colors_d[i], text=f"{int(count)}", textposition="inside"))
fig.update_layout(barmode="relative", height=200, xaxis_title="← Negative | Positive →",
                  title="Sentiment Score Distribution (Diverging)",
                  legend=dict(orientation="h", y=-0.3, x=0.5, xanchor="center"))
fig.show()

In [ ]:
# Theme Distribution
td = df2["theme"].value_counts().reset_index()
td.columns = ["Theme","Count"]
td["Pct"] = (td["Count"]/td["Count"].sum()*100).round(2)
fig = px.bar(td, x="Count", y="Theme", orientation="h", text="Pct", color="Theme",
             color_discrete_sequence=COLORS, title="Theme Distribution")
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(height=450, showlegend=False, yaxis=dict(categoryorder="total ascending"))
fig.show()

In [ ]:
# Theme × Sentiment
ct = pd.crosstab(df2["theme"], df2["sentiment"])
ct_pct = ct.div(ct.sum(axis=1), axis=0)*100
fig = px.bar(ct_pct.reset_index(), x="theme",
             y=[c for c in ct_pct.columns if c in NEG_POS],
             barmode="stack", color_discrete_map=NEG_POS,
             title="Theme × Sentiment (%)")
fig.update_layout(height=400, yaxis_title="%")
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Cohort Sentiment Analysis
for cohort in ["education", "zone", "age_range", "income_bucket"]:
    ct = pd.crosstab(df2[cohort], df2["sentiment"])
    ct_pct = ct.div(ct.sum(axis=1), axis=0)*100
    fig = px.bar(ct_pct.reset_index(), x=cohort,
                 y=[c for c in ct_pct.columns if c in NEG_POS],
                 barmode="stack", color_discrete_map=NEG_POS,
                 title=f"Sentiment by {cohort.replace('_',' ').title()} (%)")
    fig.update_layout(height=400, yaxis_title="%")
    fig.show()

In [ ]:
# Negative Reviews Deep Dive
neg_df = df2[df2["sentiment"]=="Negative"].sort_values("sentiment_score")
print(f"Negative reviews: {len(neg_df)} ({len(neg_df)/len(df2)*100:.1f}%)")
print("\nTop Negative Reviews:")
for i, (_, r) in enumerate(neg_df.head(10).iterrows(), 1):
    print(f"  {i}. [{r['rating']}⭐ | {r['sentiment_score']:.2f}] {str(r['content'])[:150]}")
    print(f"     Theme: {r['theme']}")

---
## Part 3: Summary Report

In [ ]:
import analysis as an
report = an.generate_report(df1, df2)
display(Markdown(report))
with open("report.md", "w", encoding="utf-8") as f:
    f.write(report)
print("\n✅ Report saved to report.md")